## 5.1 从通用 RL 到 LLM
如果前面几章在讲：

* 模型怎么通过 SFT 学会格式，
* 怎么通过 EI 学会把“偶然做对”固化下来，

那么这一节开始，我们终于要把视角切到 **模型在生成每一个 token 时，究竟是在做一个什么样的“强化学习决策过程”？**
### 5.1.1 名词定义
| RL 概念               | 通用含义             | LLM 里的对应物                              |
| ------------------- | ---------------- | -------------------------------------- |
| Agent               | 做决策的主体           | 当前策略模型 ( \pi_\theta )                  |
| Environment         | 接收动作、返回新状态和奖励的系统 | prompt 分布 + 文本拼接规则 + 停止规则 + 奖励模型/规则验证器 |
| State / Observation | 当前可见的局面          | 当前上下文前缀：prompt + 已生成 token             |
| Action              | 当前一步的决策          | 下一个 token                              |
| Policy              | 从状态到动作分布的映射      | LM 对词表的 next-token 分布                  |
| Reward              | 对行为好坏的标量反馈       | 规则奖励 / RM / verifier 输出的分数             |

第一，**LLM 的状态通常就是“当前前缀”**，也就是 prompt 加上目前为止生成出的 token 序列；在 decoder-only 模型的常见简化里，state 和 observation 可以近似看成同一个东西。
第二，**环境不应该只被写成“当前上下文 + 奖励函数”**，更完整一点，它还包括 prompt 采样、停止规则、可能的工具调用反馈，以及最终的奖励评估器。
```text
state s_t   = 当前文本前缀
action a_t  = 采样出的下一个 token
transition  = 把 a_t 追加到前缀上
reward      = 通常在整段回答结束后才给
```
> 这个“policy + environment”的拆法，与经典 RL formalism 和 policy optimization 教程是一致的；在 LLM 后训练里，InstructGPT 用人类偏好奖励模型，DeepSeek-R1 用规则/结果驱动的 RL，也都遵循这个框架。

只要 token (a_t) 已经确定，新状态 (s_{t+1}) 基本就是把它拼接到当前前缀上。这个“确定性转移”与很多传统 RL 环境不同：在 Atari、机器人控制、游戏世界里，环境会自己抖动、敌人会乱动、物理也会引入额外随机性；但在纯文本生成、固定 prompt、固定 verifier 的 LLM 设定里，转移本身往往很干净。这里真正的随机性，更多来自 **策略采样** 而不是环境动力学。

### 5.1.2 随机性到底从哪来
**如果 LLM 的状态转移近乎确定，那 RL 的“探索”到底体现在哪？**

答案是：主要体现在**动作采样**上。

对同一个前缀 (s_t)，模型会输出一个词表分布 (\pi_\theta(\cdot \mid s_t))。

如果你做 greedy decoding，那每次几乎都走同一条路；如果你设 temperature、top-p、top-k 去采样，那么同一个状态下就可能走出不同 token，于是整条推理链也会分叉。这个“同一状态下随机采取不同动作”的设定，正是 **stochastic policy** 的基本范式，也是 REINFORCE、VPG、PPO 这些 policy-gradient 算法的出发点。OpenAI Spinning Up 在 policy optimization 一章就是从**stochastic, parameterized policy** 开始推的。

在真实 LLM 后训练里，随机性通常至少有三层：

1. **动作采样随机性**：temperature / top-p / top-k / multinomial sampling。
2. **episode 级随机性**：每一步训练抽到哪些 prompt。
3. **奖励侧噪声**：如果奖励来自 RM、外部工具、程序执行器，反馈也可能带不确定性。

所以直接把随机性只理解成 temperature 还是不太对劲，**在纯文本、固定 prompt、固定规则 verifier 的简化设定里，给定动作后的文本转移近乎确定；但整个训练过程仍然有来自采样、数据分布和奖励评估的随机性。**

### 5.1.3 Trajectory、Return、Q、V
#### 5.1.3.1 Trajectory
很多笔记会写` Trajectory = Prompt + 完整回答`

这个说法在直觉上没问题，但还不够细。更精确地说，一条 trajectory 是：
$$
\tau = (s_0, a_0, r_0, s_1, a_1, r_1, \dots, s_T)
$$

在 LLM 里，它不是“一个大字符串”那么简单，而是：

* 第 0 步，我看到 prompt；
* 采样一个 token；
* 前缀更新；
* 再采样一个 token；
* 一直走到 EOS / stop token / max length。

所以“Prompt + 完整回答”其实只是这条轨迹的**压缩视图**。
如果后面你要理解 per-token logprob、advantage、ratio，那一定要把 trajectory 脑补成“按 token 展开的一整条决策链”。
```text
s0 = [Prompt]
a0 = "先"
s1 = [Prompt, "先"]
a1 = "设"
s2 = [Prompt, "先", "设"]
a2 = "x"
...
aT = "</answer>"
sT+1 = 终止
```

#### 5.1.3.2 Return
在很多 reasoning 任务里，奖励是**终局稀疏奖励**：

* 最终答案对：(r_T = 1)
* 最终答案错：(r_T = 0)
* 中间步骤：通常 (r_t = 0)

于是，从任意时间步 (t) 开始的 return：
$$
G_t = \sum_{t'=t}^{T} r_{t'}
$$

* 如果最后答对：所有 (G_t = 1)
* 如果最后答错：所有 (G_t = 0)

也就是说，**同一条轨迹里，每个 token 背到的是同一个最终结果**。

这导致了一个严重的 **功劳分配 (Credit Assignment) 问题**：算法会将整条序列的最终结果等同地分配给每一个 token，无法区分哪些 token 对正确答案有实质贡献，哪些是冗余甚至误导。
> 这正是 vanilla REINFORCE 在 reasoning 任务中方差极大的根本原因。因为它会把“最后撞对了”这个结果，几乎同等地归因给整条轨迹里的所有 token。

#### 5.1.3.4 折扣回报
在很多数学/代码推理任务里，我们不想鼓励模型“早点结束”，而是更关心最终是否正确，因此常常取 undiscounted return 即不折扣。

如果任务要求输出简洁（例如希望答案越短越好），也可以引入小于 1 的折扣因子。
#### 5.1.3.5 Q 与 V
**在当前前缀下，写出某个 token，到底意味着什么？**
* **(Q_\pi(s_t, a_t))**：
  现在前缀是 (s_t)，如果我这一步先强制写下 token (a_t)，之后再按当前策略继续生成，最终能拿到多大期望回报？

* **(V_\pi(s_t))**：
  现在前缀是 (s_t)，如果我什么也不强制、直接按当前策略继续生成，最终能拿到多大期望回报？

于是自然有：
$$
V_\pi(s_t) = \mathbb{E}*{a_t \sim \pi(\cdot|s_t)}[Q*\pi(s_t, a_t)]
$$
这在 LLM 里特别好理解：

* (Q)：**这个 token 选项值不值**
* (V)：**当前这条思路整体还有没有希望**
* **Advantage 其实就是：这个 token 的前途，比“在这个前缀下平均会写出来的 token”好多少。**

$$
A_\pi(s_t, a_t) = Q_\pi(s_t, a_t) - V_\pi(s_t)
$$
- 如果 (A > 0)，说明这一步比平均水平好；
- 如果 (A < 0)，说明这一步其实把你往“平均以下”的方向带了。
### 5.1.4 策略梯度
回到隔壁自学强化学习的那个笔记，更新策略分为DQN和REINFORCE两种方法。
前者作为 Value-based 方法的核心是先把 (Q(s,a)) 学准，再从中选动作。
可 LLM 的 action space 是整个词表，常常有上万到十几万个 token；而且自然语言动作空间本身还带有强组合性。ACL 2024 一篇专门研究自然语言 action space 的论文就直接指出：**自然语言动作空间会遭遇 combinatorial curse of dimensionality。**
1. **动作空间太大**：每一步都显式评估一大堆 token 的 Q 值，代价很高。
2. **序列太长**：一个回答是整串 token，真正关心的是整条序列质量，而不是孤立 token。

相比之下，policy-based 路线天然贴合语言模型本体：

* LLM 本来就输出 (\pi_\theta(a|s))
* logprob 本来就现成
* 采样本来就是生成机制的一部分
所以在 LLM 后训练里，PPO、GRPO 这类 policy-gradient / actor-critic 系方法才成了主流；InstructGPT 明确用 PPO 做 RLHF，DeepSeek-R1 也沿着 RL 驱动 reasoning 的路径推进。

然而 **“主要走 policy gradient” 不等于 “完全不用 value”。**

PPO 就是典型的 actor-critic：
policy 负责出动作分布，value network / baseline 负责减小方差。
所以更准确的说法应该是：
> **LLM 后训练主要优化 policy，但常常借助 value/baseline 来稳定训练。**

## 5.2 REINFORCE
### 5.2.1 优化目标
在强化学习里，所有算法最终都围绕一个目标：**找到一组策略参数 \(\theta^*\)，使得在策略 \(\pi_\theta\) 下智能体能够获得最高的期望回报。**
$$
J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta} \left[ R(\tau) \right]
$$
- $\theta$：策略模型（也就是 LLM）的所有可训练参数。
- $\tau$：一条完整的交互轨迹，从prompt 到 `<answer>` 开始生成到结束，包含状态、动作和奖励的序列。
- $\pi_\theta$：参数为 \(\theta\) 的策略，对于 LLM 来说就是给定前缀时输出 next-token 的概率分布。
- $R(\tau)$：轨迹 \(\tau\) 的总回报。在数学推理设定下，它通常是最后正确性奖励的 0/1 值。
- $\mathbb{E}_{\tau \sim \pi_\theta}[\cdot]$：对所有可能的轨迹取期望，这些轨迹是根据当前策略采样得到的。

我们的任务就是通过**梯度上升**来最大化 $J(\theta)$：
$$
\theta_{k+1} = \theta_k + \alpha \nabla_\theta J(\theta_k)
$$
其中 $\alpha$ 是学习率。因此，问题的核心变成了**如何计算梯度 $\nabla_\theta J(\theta)$**。

### 5.2.2 从轨迹概率到对数概率
我们现在要推导出一个可以直接从采样轨迹中估计的梯度表达式。推导过程依赖三个基础事实。
#### 5.2.2.1 轨迹的概率可以拆成一系列乘积

一条轨迹 $\tau = (s_0, a_0, s_1, a_1, \dots, s_T, a_T)$ 并不是凭空出现的。它的生成概率由三部分决定：

1. 初始状态分布 $\rho(s_0)$：告诉我们第一个状态 $s_0$ 出现的概率。在 LLM 中，$s_0$ 就是 prompt，$\rho(s_0)$ 就是训练时我们抽取 prompt 的分布。
2. 环境转移概率 $P(s_{t+1} | s_t, a_t)$：给定当前状态和动作，下一状态的条件概率。在 LLM 中，这是确定性的：$s_{t+1} = s_t \oplus a_t$（拼接 token）。
3. 策略概率 $\pi_\theta(a_t | s_t)$：在当前状态下选择某个动作的概率，这正是我们 LLM 输出的 next-token 分布。

因此，一条轨迹的联合概率可以写成：

$$
P(\tau | \theta) = \rho(s_0) \prod_{t=0}^{T} P(s_{t+1} | s_t, a_t) \, \pi_\theta(a_t | s_t)
$$
对两边取对数，乘积就变成求和：

$$
\log P(\tau | \theta) = \log \rho(s_0) + \sum_{t=0}^{T} \left[ \log P(s_{t+1} | s_t, a_t) + \log \pi_\theta(a_t | s_t) \right]
$$

后面我们会用到这个对数形式，因为在求梯度时，对数能够把乘法拆开，让只和策略相关的项单独暴露出来。
#### 5.2.2.2 对数导数技巧
这是一个纯数学技巧。我们都知道对数函数的导数公式：
$$
\frac{d}{dx} \log(f(x)) = \frac{f'(x)}{f(x)}
$$
这是链式法则的直接应用：先对 $\log(u)$ 求导得 $1/u$，再乘以 $u$ 对 $x$ 的导数 $f'(x)$。

现在，我们把这个等式**两边同时乘以 $f(x)$**，右边的 $f(x)$ 直接约掉了，就得到：
$$
f'(x) = f(x) \cdot \frac{d}{dx} \log(f(x))
$$

在机器学习中，我们的函数通常不是单变量的，而是关于参数向量 $\theta$ 的函数 $p(x|\theta)$。

这时候，导数就变成了**梯度** $\nabla_\theta$，但上面的推导**完全不变**：
$$
\nabla_\theta \log(p(x|\theta)) = \frac{\nabla_\theta p(x|\theta)}{p(x|\theta)}
$$
同样两边乘以 $p(x|\theta)$：
$$
p(x|\theta) \cdot \nabla_\theta \log(p(x|\theta)) = p(x|\theta) \cdot \frac{\nabla_\theta p(x|\theta)}{p(x|\theta)}
$$
右边约掉 $p(x|\theta)$，就得到了恒等式：
$$
\nabla_\theta p(x|\theta) = p(x|\theta) \cdot \nabla_\theta \log p(x|\theta)
$$

#### 5.2.2.3 环境项与回报对参数 $$\theta$$ 的梯度为零
在 LLM 场景中（以及多数标准 RL 设定中）：
- 初始状态分布 $\rho(s_0)$ 由 prompt 采样决定，不依赖模型参数 $\theta$，所以 $\nabla_\theta \rho(s_0) = 0$。
- 环境转移 $P(s_{t+1} | s_t, a_t)$ 由文本拼接规则确定，也不依赖 $\theta$，所以 $\nabla_\theta P(s_{t+1} | s_t, a_t) = 0$。
- 轨迹回报 $R(\tau)$ 只依赖于环境和动作序列，但计算 $R(\tau)$ 的过程同样不含参数 $\theta$（它是标量函数），因此 $\nabla_\theta R(\tau) = 0$。

这一点非常关键：当我们对 $J(\theta)$ 求梯度时，唯一会产生非零贡献的就只有策略项 $\pi_\theta$。

现在开始正式推导。
#### 5.2.2.4 正式推导
目标函数是**期望回报**：
$$
J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta} \left[ R(\tau) \right]
$$
对于连续随机变量，期望的**严格数学定义**就是积分：
> 如果一个随机变量$$X$$的概率密度函数是 $p(x)$，那么它的期望就是：
> $\mathbb{E}[X] = \int x \cdot p(x) \, dx$
在我们的场景中：
- 随机变量是**轨迹 $\tau$**
- 轨迹的概率密度函数是 $P(\tau|\theta)$（给定参数 $\theta$时，生成这条轨迹的概率）
- 我们要求期望的函数是**回报 $R(\tau)$**
所以，把期望的定义直接套进去，目标函数 $J(\theta)$ 就可以写成：
$$
J(\theta) = \int R(\tau) \cdot P(\tau|\theta) \, d\tau
$$
- 积分符号$\int$：遍历所有可能的轨迹
- $R(\tau)$：这条轨迹能获得多少回报
- $P(\tau|\theta)$：在当前参数$$\theta$$下，生成这条轨迹的概率
- $d\tau$：积分的测度，可以理解为"所有可能轨迹的集合"
> 我们把**每条轨迹的回报**乘以**生成这条轨迹的概率**，然后把所有轨迹加起来，得到的就是**平均期望回报**。
现在我们要最大化$J(\theta)$，所以需要求它对参数$\theta$的梯度$\nabla_\theta J(\theta)$。

我们对式(5.6)两边同时求梯度：
$$
\nabla_\theta J(\theta) = \nabla_\theta \left[ \int R(\tau) \cdot P(\tau|\theta) \, d\tau \right]
$$

现在，关键问题来了：**梯度算子$\nabla_\theta$能不能直接移到积分符号里面？**

答案是：**可以！** 这就是微积分中的**莱布尼茨积分法则**：
> 如果积分上下限不依赖于求导变量$\theta$，那么求导和积分的顺序可以交换。
> 在我们的场景中：
> - 积分是对**所有可能的轨迹**进行的，轨迹的集合是固定的，不依赖于参数$\theta$
> - 所以积分上下限不依赖于$\theta$
> - 因此，我们可以把梯度算子直接移到积分里面
于是式子变成：
$$
\nabla_\theta J(\theta) = \int \nabla_\theta \left[ R(\tau) \cdot P(\tau|\theta) \right] \, d\tau
$$
现在积分里面是两个函数的乘积：$R(\tau)$和$P(\tau|\theta)$。根据乘积求导法则：
$$
\nabla_\theta [A \cdot B] = (\nabla_\theta A) \cdot B + A \cdot (\nabla_\theta B)
$$

所以我们可以把积分里面的梯度展开：
$$
\nabla_\theta \left[ R(\tau) \cdot P(\tau|\theta) \right] = \left[ \nabla_\theta R(\tau) \right] \cdot P(\tau|\theta) + R(\tau) \cdot \left[ \nabla_\theta P(\tau|\theta) \right]
$$
还记得我们之前讲的**事实3**吗？
> **5.2.2.3：环境项与回报对参数$\theta$的梯度为零**
> 轨迹回报$R(\tau)$只依赖于环境和动作序列，但计算$R(\tau)$的过程同样不含参数$\theta$（它是标量函数），因此$\nabla_\theta R(\tau) = 0$。
这意味着上面展开式中的**第一项等于零**！整个乘积的梯度就只剩下第二项了：
$$
\nabla_\theta \left[ R(\tau) \cdot P(\tau|\theta) \right] = R(\tau) \cdot \nabla_\theta P(\tau|\theta)
$$
这个式子**无法直接计算**，因为$\nabla_\theta P(\tau|\theta)$是对整条轨迹概率乘积的求导，会导致计算爆炸和数值下溢。

应用对数导数技巧（事实 2），把 $\nabla_\theta P$ 替换成 $P \cdot \nabla_\theta \log P$：

$$
\nabla_\theta J(\theta) = \int R(\tau) P(\tau|\theta) \nabla_\theta \log P(\tau|\theta) \, d\tau
$$

这个积分恰好就是期望的形式：

$$
\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta} \left[ R(\tau) \nabla_\theta \log P(\tau|\theta) \right]
$$
这一步把一个**理论上存在但无法计算的积分**，变成了一个**可以通过采样来估计的期望**。
在实际训练中，我们不需要遍历所有可能的轨迹（这是不可能的），只需要：
1. 用当前策略$\pi_\theta$采样$N$条轨迹$\{\tau^{(1)}, \tau^{(2)}, \dots, \tau^{(N)}\}$
2. 对每条轨迹计算$R(\tau^{(i)}) \cdot \nabla_\theta \log P(\tau^{(i)}|\theta)$
3. 把这$N$个值加起来除以$N$，就得到了梯度的无偏估计
> 这就是**蒙特卡洛估计**，也是所有强化学习算法的基础。

里面还有一个$\nabla_\theta \log P(\tau|\theta)$需要展开。利用 **事实1：轨迹概率的对数形式**（式5.4）的对数轨迹概率：
$$
\log P(\tau|\theta) = \log \rho(s_0) + \sum_{t=0}^{T} \log P(s_{t+1}|s_t,a_t) + \sum_{t=0}^{T} \log \pi_\theta(a_t|s_t)
$$

这个式子把$\log P(\tau|\theta)$拆成了三个独立的部分：
1. $\log \rho(s_0)$：初始状态的对数概率
2. $\sum_{t=0}^{T} \log P(s_{t+1}|s_t,a_t)$：所有环境转移的对数概率之和
3. $\sum_{t=0}^{T} \log \pi_\theta(a_t|s_t)$：所有策略动作的对数概率之和

根据梯度的线性性质：**和的梯度等于梯度的和**。所以：
$$
\nabla_\theta \log P(\tau|\theta) = \nabla_\theta \log \rho(s_0) + \nabla_\theta \sum_{t=0}^{T} \log P(s_{t+1}|s_t,a_t) + \nabla_\theta \sum_{t=0}^{T} \log \pi_\theta(a_t|s_t)
$$

根据事实 3：
- $\rho(s_0)$是初始状态的分布，也就是我们训练时抽取prompt的分布。这个分布是**我们自己定义的**，和模型参数$\theta$完全无关。
- $P(s_{t+1}|s_t,a_t)$是环境转移概率，也就是给定当前状态$s_t$和动作$a_t$，下一状态$s_{t+1}$的概率。在LLM的场景中，这个转移是**完全确定性的**：下一状态就是当前状态拼接上生成的token，即$s_{t+1} = s_t \oplus a_t$。这个拼接规则是固定的，和模型参数$\theta$没有任何关系。
- $\pi_\theta(a_t|s_t)$是策略在状态$s_t$下选择动作$a_t$的概率，也就是LLM在给定前缀$s_t$时输出token$a_t$的概率。这个概率完全由模型参数$\theta$决定。
同样根据梯度的线性性质，和的梯度等于梯度的和：
$$
\nabla_\theta \sum_{t=0}^{T} \log \pi_\theta(a_t|s_t) = \sum_{t=0}^{T} \nabla_\theta \log \pi_\theta(a_t|s_t)
$$
现在，我们把这三项的梯度结果加起来：
$$
\nabla_\theta \log P(\tau|\theta) = 0 + 0 + \sum_{t=0}^{T} \nabla_\theta \log \pi_\theta(a_t|s_t)
$$

也就是：
$$
\nabla_\theta \log P(\tau|\theta) = \sum_{t=0}^{T} \nabla_\theta \log \pi_\theta(a_t|s_t)
$$
这一步太重要了！它告诉我们：
> 整条轨迹概率的对数的梯度，**等于这条轨迹中每个token的对数概率的梯度之和**。

它把一个复杂的、对乘积的求导，变成了一个简单的、对求和的求导。

现在，我们把式(5.10)代回式(5.9)：
$$
\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta} \left[ R(\tau) \cdot \sum_{t=0}^{T} \nabla_\theta \log \pi_\theta(a_t|s_t) \right]
$$

我们把$R(\tau)$移到求和符号里面（因为$R(\tau)$是整条轨迹的总回报，和时间步$t$无关），就得到了著名的**REINFORCE策略梯度公式**：
$$
\nabla_\theta J(\pi_\theta) = \mathbb{E}_{\tau \sim \pi_\theta} \left[ \sum_{t=0}^{T} \nabla_\theta \log \pi_\theta(a_t|s_t) \, R(\tau) \right]
$$
式子 (5.11) 可以拆成两部分看：

- $$\nabla_\theta \log \pi_\theta(a_t|s_t)$$ 是一个向量，它告诉我们**怎样改变参数 $$\theta$$ 才能让当前动作 $$a_t$$ 在状态 $$s_t$$ 下的概率增加**。
- $$R(\tau)$$ 是一个标量权重，它告诉我们**这个动作所处的整条轨迹最终得了多少分**。

所以 REINFORCE 的核心思想是：**如果一条轨迹的总回报高，就把这条轨迹里所有出现的动作的概率往上拉；如果回报低，拉高的幅度就小（或甚至不拉高，视具体权重而定）。** 每个动作的梯度贡献被整条轨迹的最终结果所缩放。

注意，这个权重是整条轨迹共享的——也就是说，同一个轨迹里，不管是开头的 token 还是结尾的 token，它们得到的"鼓励"都是按照相同的 $$R(\tau)$$ 来缩放的。
### 5.2.3 梯度估计
这个式子有一个**致命的实践问题**：它是一个对**所有可能轨迹**的期望。在数学推理任务中，一条轨迹可能包含几十甚至上百个token，所有可能的轨迹数量是**词表大小的T次方**——这是一个天文数字，我们永远不可能遍历所有轨迹来计算这个期望。

这时候，我们就需要用到 **统计学中最强大的工具之一：蒙特卡洛估计（Monte Carlo Estimation）**，它基于 **大数定律**。

我们不需要知道所有可能的轨迹，只需要从当前策略$\pi_\theta$中采样$N$条真实的轨迹，然后计算这些轨迹的梯度的平均值，这个平均值就是真实梯度的一个**无偏估计**。

$$
\hat{g} = \frac{1}{N} \sum_{i=1}^{N} \sum_{t=0}^{T_i} \nabla_\theta \log \pi_\theta(a_t^{(i)} | s_t^{(i)}) \, R(\tau^{(i)})
$$

| 符号 | 含义 |
|------|----------------------------------|
| $\hat{g}$ | 梯度估计值（我们实际用来更新参数的梯度） |
| $N$ | 一个batch中采样的轨迹总数（CS336推荐单卡batch size=4） |
| $i$ | 轨迹的索引（从1到N，代表batch中的第i条轨迹） |
| $\tau^{(i)}$ | batch中的第i条轨迹（对应一个数学问题+模型生成的完整回答） |
| $T_i$ | 第i条轨迹的长度（不同问题的回答长度可能不同，所以用$T_i$而不是固定的$T$） |
| $t$ | 轨迹中的时间步索引（从0到$T_i$，代表回答中的第t个token） |
| $s_t^{(i)}$ | 第i条轨迹中第t步的状态（也就是生成第t个token之前的所有前缀文本） |
| $a_t^{(i)}$ | 第i条轨迹中第t步的动作（也就是模型实际生成的第t个token） |
| $R(\tau^{(i)})$ | 第i条轨迹的总回报（在CS336中就是0或1：回答正确得1，错误得0） |
| $\frac{1}{N}$ | 对batch中所有轨迹的梯度取平均 |

假设我们现在有一个batch，$N=2$（为了简单，用2条轨迹举例）：

- **轨迹1（$\tau^{(1)}$）**：
  - 问题：1+1=?
  - 模型回答：2
  - 回答长度：$T_1=1$
  - 答案正确：$R(\tau^{(1)})=1$

- **轨迹2（$\tau^{(2)}$）**：
  - 问题：2+3=?
  - 模型回答：6
  - 回答长度：$T_2=1$
  - 答案错误：$R(\tau^{(2)})=0$

现在我们来计算这个batch的梯度估计$\hat{g}$：

首先计算第一条轨迹的梯度：
$$
\sum_{t=0}^{0} \nabla_\theta \log \pi_\theta(a_t^{(1)} | s_t^{(1)}) \cdot R(\tau^{(1)}) = \nabla_\theta \log \pi_\theta(\text{"2"} | \text{"1+1=?"}) \cdot 1
$$

然后计算第二条轨迹的梯度：
$$
\sum_{t=0}^{0} \nabla_\theta \log \pi_\theta(a_t^{(2)} | s_t^{(2)}) \cdot R(\tau^{(2)}) = \nabla_\theta \log \pi_\theta(\text{"6"} | \text{"2+3=?"}) \cdot 0 = 0
$$

最后对两条轨迹的梯度取平均：
$$
\hat{g} = \frac{1}{2} \left[ \nabla_\theta \log \pi_\theta(\text{"2"} | \text{"1+1=?"}) + 0 \right] = \frac{1}{2} \nabla_\theta \log \pi_\theta(\text{"2"} | \text{"1+1=?"})
$$
得到梯度估计$\hat{g}$之后，我们就可以用它来更新模型参数了。

和SFT中使用**梯度下降**（最小化损失）不同，在RL中我们使用**梯度上升**（最大化期望回报）：
$$
\theta \leftarrow \theta + \alpha \hat{g}
$$
> CS336推荐RL阶段的学习率是$5e-7$，比SFT的$5e-6$小一个数量级，就是为了应对高方差的问题
## 5.3 降低方差：Baseline 与 Advantage 函数
这个梯度估计$\hat{g}$有两个非常重要的性质，直接决定了REINFORCE算法的行为：
1. 无偏性（Unbiased）
**样本平均是真实期望的无偏估计**。也就是说：
$$
\mathbb{E}[\hat{g}] = \nabla_\theta J(\pi_\theta)
$$

这意味着，如果我们采样足够多的batch，那么所有梯度估计的平均值会精确等于真实梯度。这是一个非常好的性质——它保证了我们的训练方向在统计上是正确的。

2. 高方差（High Variance）
这是REINFORCE算法最大的问题。梯度估计$\hat{g}$的方差非常大，也就是说：
> 即使我们用同一个策略$\pi_\theta$采样两个不同的batch，得到的梯度估计$\hat{g}_1$和$\hat{g}_2$可能会相差非常大。

在数学推理任务中，这个问题尤其严重，因为回报$R(\tau)$只有0和1两个值：
- 如果一个batch中恰好有很多正确的回答，那么梯度会很大，参数会更新很多
- 如果一个batch中恰好没有正确的回答，那么梯度会是0，参数完全不更新
假设我们正在训练一个数学推理模型，当前策略的正确率是50%。我们用batch size=4进行训练，那么一个batch中可能出现的正确回答数量是0、1、2、3、4。

根据REINFORCE的梯度估计公式：
$$
\hat{g} = \frac{1}{4} \sum_{i=1}^{4} \sum_{t=0}^{T_i} \nabla_\theta \log \pi_\theta(a_t^{(i)} | s_t^{(i)}) \cdot R(\tau^{(i)})
$$

我们来看看不同情况下的梯度：
- 如果batch中**0个正确**：所有$R(\tau^{(i)})=0$，$\hat{g}=0$，参数完全不更新
- 如果batch中**1个正确**：$\hat{g} = \frac{1}{4} \cdot g_{\text{correct}}$
- 如果batch中**2个正确**：$\hat{g} = \frac{2}{4} \cdot g_{\text{correct}}$
- 如果batch中**3个正确**：$\hat{g} = \frac{3}{4} \cdot g_{\text{correct}}$
- 如果batch中**4个正确**：$\hat{g} = \frac{4}{4} \cdot g_{\text{correct}}$
仅仅因为采样的随机性，同一个策略在不同batch上得到的梯度估计值，**大小相差了无穷多倍**！这种剧烈的波动会导致：
- 模型参数像过山车一样来回震荡
- 好不容易学到的正确行为可能被一个坏batch完全抹掉
- 训练过程极不稳定，很难收敛

为了解决 REINFORCE 高方差的问题，最经典的方法就是引入一个 **基线 (baseline)**，从而把回报替换为 **优势 (advantage)**。
### 5.3.1 baseline
Baseline的思想非常朴素：**我们不关心一个动作能获得多少绝对回报，只关心它比平均水平好多少**。

设想一下：
- 如果当前策略的平均正确率是50%，那么一个获得1分的回答其实只是达到了平均水平，不应该给它太大的奖励
- 如果当前策略的平均正确率只有10%，那么一个获得1分的回答就是远超平均水平的优秀表现，应该给它很大的奖励
- 反之，如果当前策略的平均正确率是90%，那么一个获得0分的回答就是远低于平均水平的糟糕表现，应该给它很大的惩罚
设想我们可以为每个状态 $s_t$ 估计一个"平均预期回报" $b(s_t)$。当我们评估动作 $a_t$ 时，不再看绝对回报 $R(\tau)$，而是看 **这个动作带来的回报比平均预期高多少**，即 $R(\tau) - b(s_t)$。这样，如果一个动作只是达到了平均水平，它的梯度权重就接近零，只有明显好于或差于平均水平的动作才会被强调，从而降低估计的波动。

修改后的梯度形式为：

$$
B = \mathbb{E}_{\tau \sim \pi_\theta} \left[ \sum_{t=0}^{T} \nabla_\theta \log \pi_\theta(a_t|s_t) \left( R(\tau) - b(s_t) \right) \right]
$$
### 5.3.2 无偏性
**只要基线$b(s_t)$只依赖于状态$s_t$（而不依赖于动作$a_t$），那么加入它不会改变梯度的期望值**。

也就是说，我们在**完全不改变训练方向**的前提下，**大幅降低了梯度估计的方差**。这是一个鱼和熊掌可以兼得的改进。

证明的核心是一个被称为"得分函数性质"的基本数学事实。我们先证明这个性质，再用它来证明Baseline的无偏性。

**得分函数性质**：对于任意状态$s$和任意策略$\pi_\theta$，恒有：
$$
\mathbb{E}_{a \sim \pi_\theta(\cdot|s)} \left[ \nabla_\theta \log \pi_\theta(a|s) \right] = 0
$$

根据概率的基本性质，对于任意状态$s$，所有可能动作的概率之和恒为1：
$$
\sum_{a} \pi_\theta(a|s) = 1
$$

现在，我们对等式两边同时关于$\theta$求梯度：
$$
\nabla_\theta \sum_{a} \pi_\theta(a|s) = \nabla_\theta 1
$$

- 左边：和的梯度等于梯度的和
- 右边：常数的梯度等于0

所以：
$$
\sum_{a} \nabla_\theta \pi_\theta(a|s) = 0
$$

现在，我们应用对数导数技巧：
$$
\nabla_\theta \pi_\theta(a|s) = \pi_\theta(a|s) \cdot \nabla_\theta \log \pi_\theta(a|s)
$$
代入上式：
$$
\sum_{a} \pi_\theta(a|s) \cdot \nabla_\theta \log \pi_\theta(a|s) = 0
$$
而左边恰好就是期望的定义！所以：
$$
\mathbb{E}_{a \sim \pi_\theta(\cdot|s)} \left[ \nabla_\theta \log \pi_\theta(a|s) \right] = 0
$$

证毕。

这个性质非常重要，它是所有策略梯度算法的数学基石。

现在我们来看式(5.13)中的Baseline项：
$$
\mathbb{E}_{\tau \sim \pi_\theta} \left[ \sum_{t=0}^{T} \nabla_\theta \log \pi_\theta(a_t|s_t) \cdot b(s_t) \right]
$$

我们要证明这个期望等于零。

首先，根据期望的线性性质，和的期望等于期望的和：
$$
\mathbb{E}_{\tau} \left[ \sum_{t} \dots \right] = \sum_{t} \mathbb{E}_{\tau} \left[ \dots \right]
$$

所以我们可以把求和符号移到期望外面：
$$
\sum_{t=0}^{T} \mathbb{E}_{\tau \sim \pi_\theta} \left[ \nabla_\theta \log \pi_\theta(a_t|s_t) \cdot b(s_t) \right]
$$

现在，我们把对整条轨迹$\tau$的期望，分解成对状态$s_t$和动作$a_t$的条件期望：
$$
\sum_{t=0}^{T} \mathbb{E}_{s_t \sim \pi_\theta} \left[ b(s_t) \cdot \mathbb{E}_{a_t \sim \pi_\theta(\cdot|s_t)} \left[ \nabla_\theta \log \pi_\theta(a_t|s_t) \bigg| s_t \right] \right]
$$
注意到内层期望是在给定状态$s_t$的条件下，对动作$a_t$取期望。根据我们刚刚证明的得分函数性质，这个内层期望等于零！
$$
\mathbb{E}_{a_t \sim \pi_\theta(\cdot|s_t)} \left[ \nabla_\theta \log \pi_\theta(a_t|s_t) \bigg| s_t \right] = 0
$$

所以整个式子就变成了：
$$
\sum_{t=0}^{T} \mathbb{E}_{s_t} \left[ b(s_t) \cdot 0 \right] = \sum_{t=0}^{T} 0 = 0
$$
现在我们把式(5.13)展开：
$$
B = \mathbb{E}_{\tau} \left[ \sum_{t} \nabla_\theta \log \pi_\theta \cdot R(\tau) \right] - \mathbb{E}_{\tau} \left[ \sum_{t} \nabla_\theta \log \pi_\theta \cdot b(s_t) \right]
$$

我们刚刚证明了第二项等于零，所以：
$$
B = \mathbb{E}_{\tau} \left[ \sum_{t} \nabla_\theta \log \pi_\theta \cdot R(\tau) \right] = \nabla_\theta J(\theta)
$$

这就证明了：**加入Baseline后的梯度$B$，和原来的REINFORCE梯度$\nabla_\theta J(\theta)$是完全相等的**。

也就是说，Baseline是一个**零成本的改进**：它完全不改变梯度的期望值（无偏），但能大幅降低梯度估计的方差。
### 5.3.3 选择$V^\pi(s)$作为最优基线
理论上，任何只依赖于状态$s_t$的函数都可以作为Baseline。但在实践中，我们总是选择 **状态价值函数$V^\pi(s)$** 作为Baseline，因为它是 **能最小化梯度估计方差的最优基线**。

$$
V^\pi(s_t) = \mathbb{E}_{\tau \sim \pi_\theta} [R(\tau) | s_t]
$$

即当模型已经生成了前缀文本$s_t$时，最终回答正确的概率。
从方差的角度来看，梯度估计的方差为：
$$
\text{Var}(\hat{g}) = \mathbb{E}[\hat{g}^2] - (\mathbb{E}[\hat{g}])^2
$$

因为$\hat{g}$是无偏的，所以$(\mathbb{E}[\hat{g}])^2 = (\nabla_\theta J(\theta))^2$是一个常数。因此，最小化方差等价于最小化$\mathbb{E}[\hat{g}^2]$。

可以证明，当且仅当基线$b(s_t) = V^\pi(s_t)$时，$\mathbb{E}[\hat{g}^2]$取得最小值。数学证明我已经力竭了，就不给出详细的证明了。

现在，我们把优势函数代入梯度估计公式，就得到了**带Baseline的REINFORCE算法**的最终梯度估计：
$$
\hat{g} = \frac{1}{N} \sum_{i=1}^{N} \sum_{t=0}^{T_i} \nabla_\theta \log \pi_\theta(a_t^{(i)}|s_t^{(i)}) \cdot A^\pi(s_t^{(i)}, a_t^{(i)})
$$

或者用近似形式：
$$
\hat{g} = \frac{1}{N} \sum_{i=1}^{N} \sum_{t=0}^{T_i} \nabla_\theta \log \pi_\theta(a_t^{(i)}|s_t^{(i)}) \left( R(\tau^{(i)}) - V^\pi(s_t^{(i)}) \right)
$$

## 5.4 On-policy与Off-policy
### 5.4.1 On-policy
我们的REINFORCE梯度是这样定义的：
$$
\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta} \left[ \sum_{t=0}^{T} \nabla_\theta \log \pi_\theta(a_t|s_t) \cdot A(s_t,a_t) \right]
$$

注意这个期望的下标：$\tau \sim \pi_\theta$。这意味着：
> 这个梯度是**在当前策略$\pi_\theta$生成的轨迹分布上**的期望。
换句话说，这个梯度的定义本身就**绑定了采样数据的分布**。如果我们用来计算梯度的轨迹不是来自$\pi_\theta$，而是来自另一个不同的策略$\pi_{\theta_{old}}$，那么我们计算出来的就不是$\nabla_\theta J(\theta)$，而是另一个完全不同的梯度。

假设我们有一个非常简单的任务：模型需要输出"是"或"否"来回答一个问题。
- **初始策略$\pi_{\theta_0}$**：输出"是"的概率是0.5，输出"否"的概率是0.5
- 我们用$\pi_{\theta_0}$采样了一条轨迹：模型输出了"是"，并且回答正确，获得回报$R=1$
- 根据REINFORCE，我们计算梯度并更新参数，得到新策略$\pi_{\theta_1}$
- **新策略$\pi_{\theta_1}$**：输出"是"的概率变成了0.9，输出"否"的概率变成了0.1

现在问题来了：我们还能再用刚才那条"输出是"的轨迹来更新$\pi_{\theta_1}$吗？

答案是：**不能**！因为：
- 在$\pi_{\theta_0}$下，这条轨迹的概率是0.5，它对梯度的贡献是$\nabla_\theta \log(0.5) \cdot 1$
- 在$\pi_{\theta_1}$下，这条轨迹的概率是0.9，它对梯度的贡献是$\nabla_\theta \log(0.9) \cdot 1$

这两个梯度是**完全不同的**！如果我们用旧数据来更新新策略，我们就是在计算一个错误的梯度，训练方向就会跑偏。

所以核心在于，旧数据没用了。

### 5.4.2 LLM中On-policy的灾难性低效
在传统的强化学习任务中（比如Atari游戏），On-policy（以REINFORCE、A2C、A3C、Vanilla PPO为代表）并不是一个太大的问题，因为生成一条轨迹（玩一局游戏）非常快，比一次梯度计算快得多。

但在大语言模型的后训练中：
1. **推理比训练慢得多**：生成一条包含100个token的回答，需要模型前向传播100次；而一次梯度更新只需要一次前向传播和一次反向传播。在现代GPU上，生成的耗时通常是梯度更新的5-10倍。
2. **算力浪费严重**：如果每进行一次梯度更新就要抛弃全部旧数据，那么90%以上的算力都被浪费在了生成数据上，而不是用来更新模型。
3. **小更新下的过度保守**：实际上，模型在一次小的梯度更新后，行为变化非常小。新旧策略的分布差异微乎其微，旧数据的"偏差"几乎可以忽略不计。完全丢弃这些数据是对宝贵计算资源的巨大浪费。
| 特性 | On-policy | Off-policy |
|------|-----------|------------|
| 核心定义 | 学习策略 = 采样策略 | 学习策略 ≠ 采样策略 |
| 数据复用 | 不能复用旧数据，每次更新后必须重新采样 | 可以无限复用旧数据，数据效率极高 |
| 学习目标 | 当前策略的价值 | 最优策略的价值 |
| 代表算法 | REINFORCE, A2C, A3C, GRPO | Q-Learning, DQN, SAC, TD3, DPO |
| 优点 | 实现简单，训练稳定，无偏 | 数据效率高，能利用专家数据和离线数据 |
| 缺点 | 数据效率极低，算力浪费严重 | 实现复杂，容易引入偏差和方差 |
| 适用场景 | 数据生成成本低的场景（如游戏） | 数据生成成本高的场景（如机器人、LLM） |
> 这里小声BB一句，比如说，Q-Learning用来更新策略的$Q^*(s,a)$是一个**客观存在的量**，和你用什么策略来收集数据完全无关。无论你是用随机策略、专家策略还是其他任何策略来采样$(s,a,r,s')$，只要你有足够多的样本，Q-Learning最终都会收敛到同一个$Q^*(s,a)$
> 而REINFORCE的梯度定义本身就绑定了采样数据必须来自当前正在学习的策略。
> 用一句话来说的话，**（策略）梯度是主观的，（最优）价值是客观的**。

这就是为什么我们需要Off-policy算法（以Q-Learning、DQN、DDPG/TD3/SAC为代表）：**允许我们用旧策略采样的数据来优化新策略，让一批数据可以反复利用多次**。
### 5.4.3 重要性采样
假设我们想估计一个不均匀骰子的期望值。这个骰子掷出6的概率是0.5，掷出其他数字的概率都是0.1。所以它的期望值是：
$$
\mathbb{E}_{X \sim P}[X] = 1 \times 0.1 + 2 \times 0.1 + 3 \times 0.1 + 4 \times 0.1 + 5 \times 0.1 + 6 \times 0.5 = 4.5
$$

但现在我们只有一个均匀的骰子（每个数字的概率都是1/6），我们怎么用这个均匀骰子来估计不均匀骰子的期望值呢？

这就是重要性采样的用武之地。我们可以用均匀骰子采样，然后给每个样本乘以一个**重要性权重**$\frac{P(x)}{Q(x)}$：
$$
\mathbb{E}_{X \sim P}[X] = \mathbb{E}_{X \sim Q} \left[ X \cdot \frac{P(x)}{Q(x)} \right]
$$
验证一下：
$$
\begin{align*}
\mathbb{E}_{X \sim Q} \left[ X \cdot \frac{P(x)}{Q(x)} \right] &= \sum_{x=1}^6 x \cdot \frac{P(x)}{Q(x)} \cdot Q(x) \\
&= \sum_{x=1}^6 x \cdot P(x) \\
&= \mathbb{E}_{X \sim P}[X]
\end{align*}
$$
核心思想：**通过给每个样本乘以一个权重，来修正两个分布之间的差异**。
#### 数学证明
对于任意两个分布$P(x)$和$Q(x)$，只要$Q(x) > 0$对于所有$P(x) > 0$的$x$成立，那么恒有：
$$
\mathbb{E}_{x \sim P}[f(x)] = \int f(x) P(x) dx = \int f(x) \frac{P(x)}{Q(x)} Q(x) dx = \mathbb{E}_{x \sim Q} \left[ f(x) \cdot \frac{P(x)}{Q(x)} \right]
$$

其中$\frac{P(x)}{Q(x)}$就是**重要性权重**。
#### 策略梯度
现在我们把重要性采样应用到策略梯度中。我们的问题是：
- 我们有从旧策略$\pi_{\theta_{old}}$采样得到的轨迹$\tau \sim \pi_{\theta_{old}}$
- 我们想估计新策略$\pi_\theta$的期望回报梯度$\nabla_\theta J(\theta)$

根据重要性采样的原理，我们可以把在$\pi_\theta$下的期望，转换成在$\pi_{\theta_{old}}$下的期望，乘以重要性权重$\frac{P(\tau|\theta)}{P(\tau|\theta_{old})}$：
$$
\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta} \left[ \sum_{t=0}^{T} \nabla_\theta \log \pi_\theta(a_t|s_t) \cdot A(s_t,a_t) \right]
$$
$$
= \mathbb{E}_{\tau \sim \pi_{\theta_{old}}} \left[ \frac{P(\tau|\theta)}{P(\tau|\theta_{old})} \cdot \sum_{t=0}^{T} \nabla_\theta \log \pi_\theta(a_t|s_t) \cdot A(s_t,a_t) \right]
$$

现在我们需要计算轨迹概率的比值$\frac{P(\tau|\theta)}{P(\tau|\theta_{old})}$。我们之前已经知道，一条轨迹的概率可以写成：
$$
P(\tau|\theta) = \rho(s_0) \prod_{t=0}^{T} P(s_{t+1}|s_t,a_t) \cdot \pi_\theta(a_t|s_t)
$$
所以新旧策略下轨迹概率的比值为：
$$
\frac{P(\tau|\theta)}{P(\tau|\theta_{old})} = \frac{\rho(s_0) \prod_{t=0}^{T} P(s_{t+1}|s_t,a_t) \cdot \pi_\theta(a_t|s_t)}{\rho(s_0) \prod_{t=0}^{T} P(s_{t+1}|s_t,a_t) \cdot \pi_{\theta_{old}}(a_t|s_t)}
$$
注意到分子和分母中的**初始状态分布$\rho(s_0)$和环境转移概率$P(s_{t+1}|s_t,a_t)$是完全相同的**，它们都不依赖于策略参数$\theta$。所以它们可以精确约掉！

于是，轨迹概率的比值就简化成了**所有时间步的动作概率的乘积**：
$$
\frac{P(\tau|\theta)}{P(\tau|\theta_{old})} = \prod_{t=0}^{T} \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{old}}(a_t|s_t)}
$$
它告诉我们，轨迹概率的比值只和策略有关，和环境无关。

现在我们把这个乘积形式的重要性权重代回梯度公式：
$$
\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_{\theta_{old}}} \left[ \left( \prod_{t'=0}^{T} \frac{\pi_\theta(a_{t'}|s_{t'})}{\pi_{\theta_{old}}(a_{t'}|s_{t'})} \right) \cdot \sum_{t=0}^{T} \nabla_\theta \log \pi_\theta(a_t|s_t) \cdot A(s_t,a_t) \right]
$$
这个式子看起来很复杂，但我们可以把它改写一下。注意到对于求和中的每一项$t$，乘积$\prod_{t'=0}^{T}$可以拆成$\prod_{t'=0}^{t} \cdot \prod_{t'=t+1}^{T}$。但在实践中，我们通常做一个非常关键的近似：**忽略远期乘积，只保留到当前时间步$t$的乘积**。

也就是说，我们近似：
$$
\prod_{t'=0}^{T} \frac{\pi_\theta(a_{t'}|s_{t'})}{\pi_{\theta_{old}}(a_{t'}|s_{t'})} \approx \prod_{t'=0}^{t} \frac{\pi_\theta(a_{t'}|s_{t'})}{\pi_{\theta_{old}}(a_{t'}|s_{t'})}
$$

这个近似的直觉是：**当前时间步$t$的梯度，主要受之前时间步的动作概率的影响，而受之后时间步的动作概率的影响很小**。

在很多现代RLHF算法中（包括PPO和GRPO），我们甚至会做更进一步的近似：**完全忽略乘积，只使用当前时间步$t$的单步比率**。

定义每个时间步的**概率比率（ratio）**：
$$
\rho_t = \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{old}}(a_t|s_t)}
$$

于是，梯度公式就简化成了：
$$
\hat{g}_{\text{off}} = \mathbb{E}_{\tau \sim \pi_{\theta_{old}}} \left[ \sum_{t=0}^{T} \rho_t \cdot \nabla_\theta \log \pi_\theta(a_t|s_t) \cdot A(s_t,a_t) \right]
$$
这个近似看起来很粗糙，但在实践中效果却非常好，尤其是在LLM的场景中。原因有两个：
1. **小更新假设**：我们每次只对策略做非常小的更新，所以$\pi_\theta$和$\pi_{\theta_{old}}$非常接近，$\rho_t \approx 1$，乘积的影响很小。
2. **终局奖励假设**：在数学推理等终局奖励的任务中，优势函数$A(s_t,a_t)$对于所有时间步都是相同的（都等于$R(\tau) - b$），所以远期乘积的影响会被平均掉。

#### 方差爆炸
我们来考虑一个极端情况：
- 旧策略$\pi_{\theta_{old}}$对某个动作$a_t$的概率是$10^{-4}$（几乎不会选择这个动作）
- 新策略$\pi_\theta$对这个动作的概率是$0.1$（非常偏好这个动作）
- 那么单步比率$\rho_t = \frac{0.1}{10^{-4}} = 1000$

如果一条轨迹中有10个这样的动作，那么总的重要性权重就是$1000^{10} = 10^{30}$！这是一个天文数字。

一旦出现这样的极端权重，梯度更新的幅度会被放大到完全失控的程度：
- 模型参数会发生巨大的跳跃
- 策略会瞬间崩溃
- 训练会完全无法恢复
这就是 **分布偏移（Distribution Shift）** 带来的问题：当新旧策略差异较大时，重要性权重会变得非常大，导致梯度估计的方差爆炸。

为了解决它，PPO 引入了 **截断 (clipping)** 机制，显式限制比率远离 1 的程度，在利用 off-policy 数据效率的同时，牢牢控制住方差。